In [1]:
!pip install unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.7/192.7 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 80.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 61.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
import torch
import json
from unsloth import FastLanguageModel
import pandas as pd
import re


class FinancialCrimeClassifier:
    fourbit_models = [
        "unsloth/Meta-Llama-3.1-8B-bnb-4bit",
        "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
        "unsloth/Meta-Llama-3.1-70B-bnb-4bit",
        "unsloth/Meta-Llama-3.1-405B-bnb-4bit",
        "unsloth/Mistral-Small-Instruct-2409",
        "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
        "unsloth/Llama-3.2-1B-Instruct-bnb-4bit",
        "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
        "unsloth/Llama-3.3-70B-Instruct-bnb-4bit",
    ]

    eightbit_models = [
        "unsloth/gemma-3-4b-it",
        "unsloth/gemma-3-12b-it",
        "unsloth/gemma-3-27b-it"
    ]

    def __init__(
        self,
        model_name="unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
        max_seq_length=20000,
        dtype=None,
        device=None,
    ):
        self.model_name = model_name
        self.max_seq_length = max_seq_length
        self.dtype = dtype
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")

        self._validate_model()
        self.model, self.tokenizer = self._load_model()
        self._patch_chat_template_if_missing()

    def _validate_model(self):
        if self.model_name not in self.fourbit_models + self.eightbit_models:
            raise ValueError(
                f"Invalid model '{self.model_name}'. Please choose from:\n" +
                "\n".join(self.fourbit_models + self.eightbit_models)
            )

    def _load_model(self):
        print(f"Loading model: {self.model_name}")
        load_in_4bit = self.model_name in self.fourbit_models
        load_in_8bit = self.model_name in self.eightbit_models

        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=self.model_name,
            max_seq_length=self.max_seq_length,
            dtype=self.dtype,
            load_in_4bit=load_in_4bit,
            load_in_8bit=load_in_8bit,
        )
        model.eval()
        return model, tokenizer

    def _patch_chat_template_if_missing(self):
        if not getattr(self.tokenizer, "chat_template", None):
            print("Patching missing chat template...")

        if "gemma-3" in self.model_name:
            print("Overriding Gemma 3 chat template for compatibility...")
            self.tokenizer.chat_template = (
                "{% for message in messages %}"
                "{% if message['role'] == 'system' %}"
                "<|start|>system\n{{ message['content'][0]['text'] }}<|end|>\n"
                "{% elif message['role'] == 'user' %}"
                "<|start|>user\n{{ message['content'][0]['text'] }}<|end|>\n"
                "{% elif message['role'] == 'assistant' %}"
                "<|start|>assistant\n{{ message['content'][0]['text'] }}<|end|>\n"
                "{% endif %}"
                "{% endfor %}"
                "<|start|>assistant\n"
            )

    def _build_prompt(self, text):
        return [
            {
                "role": "system",
                "content": "You are a financial crime expert AI. Respond only in valid JSON."
            },
            {
                "role": "user",
                "content": f"""Analyze the following news article and return this exact JSON format:
{{
  \"adverse_probability\": float (0 to 1) Probability that it is adverse news related to financial crime or sanctions or scandals,
  \"financial_crime_relevance\": float (0 to 1),
  \"related_topics\": list of relevant topics from:
    [\"Money Laundering\", \"Terrorist Financing\", \"Sanctions Violations\", \"Fraud\", \"Tax Evasion\", \"Bribery and Corruption\", \"Insider Trading\", \"Ponzi and Pyramid Schemes\", \"Trade-Based Money Laundering\"]
}}

News:
\"\"\"{text}\"\"\"
Only return JSON — no explanation.
"""
            }
        ]

    def classify(self, text, return_json=True):
        messages = self._build_prompt(text)

        # Gemma 3 models require content to be a list of {"type": "text", "text": ...}
        if "gemma-3" in self.model_name:
            for m in messages:
                if isinstance(m["content"], str):
                    m["content"] = [{"type": "text", "text": m["content"]}]

        inputs = self.tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            return_tensors="pt"
        ).to(self.device)

        attention_mask = inputs != self.tokenizer.pad_token_id

        with torch.no_grad():
            outputs = self.model.generate(
                inputs,
                attention_mask=attention_mask,
                max_new_tokens=512,
                temperature=0.2  # To keep it more deterministic
            )

        response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)

        try:
            line_num = next(i for i, line in enumerate(response.splitlines())
                            if "Only return JSON" in line)
            json_lines = [line for i, line in enumerate(response.splitlines()) if i > line_num] # This is the part after the end of prompt
            # To ensure this works for Gemma, we need to explicitly find the indices of '{' and '}' respectively
            start_json_index = next(i for i, line in enumerate(json_lines)
                            if "{" in line)
            end_json_index = next(i for i, line in enumerate(json_lines)
                            if "}" in line)

            json_lines_final = json_lines[start_json_index:end_json_index + 1]

            json_text = ''.join(json_lines_final)

            # Remove hypothetical tokens
            json_text = re.sub(r'\(hypothetical\)', '', json_text)

            # Fix malformed related_topics list ending in }
            json_text = re.sub(
                r'"related_topics"\s*:\s*\[([^\]]*?)\}',  # matches until a closing brace
                lambda m: f'"related_topics": [{m.group(1).strip()}]',  # replace with proper array
                json_text
            )

            # Remove trailing commas before closing brackets/braces
            json_text = re.sub(r',\s*(\]|\})', r'\1', json_text)

            # Ensure all brackets and braces are balanced
            open_brackets = json_text.count('[')
            close_brackets = json_text.count(']')
            if open_brackets > close_brackets:
                json_text += ']' * (open_brackets - close_brackets)

            open_braces = json_text.count('{')
            close_braces = json_text.count('}')
            if open_braces > close_braces:
                json_text += '}' * (open_braces - close_braces)

            #print(json_text)
            final_json = json.loads(json_text)

        except Exception as e:
            final_json = {
                "error": f"JSON parse error: {e}",
                "raw_output": response
            }

        return final_json


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Failed to patch Gemma3ForConditionalGeneration.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [4]:
def normalize_related_topics(val):
    if isinstance(val, list):
        return val
    elif pd.isnull(val):
        return None
    else:
        return [val]

def classify_by_llm(articles, model_names = None):
    '''
    Expects articles to be a list-like object and model_names to also be either string or list of string specifying the model types
    '''

    # Check for valid values if not we default to these 2 models
    if (model_names == None) or (type(model_names) not in ['str','list']):
        model_names = ['unsloth/mistral-7b-instruct-v0.3-bnb-4bit','unsloth/Llama-3.2-3B-Instruct-bnb-4bit']

    final_prediction_df = pd.DataFrame()

    for index, model in enumerate(model_names):
        # Calls the FinancialCrimeClassifier class and predicts based on that.
        classifier = FinancialCrimeClassifier(model_name= model)

        # Initialise result_lst
        result_lst = []

        # Loop through each to give this result
        for i, article in enumerate(articles):

          if len(article) == 0:
            result = {
                      "adverse_probability": None,
                      "financial_crime_relevance": None,
                      "related_topics": []
                    }

            # Update result_lst
            result_lst.append(result)
            continue

          # Collect the result
          result = classifier.classify(article)
          # print(json.dumps(result, indent=2))

          # Update result_lst
          result_lst.append(result)

          if i % 50 == 0:
            print('{} articles processed'.format(i+1))

        prediction_df = pd.DataFrame(result_lst)
        prediction_df['Model'] = model

        # Interim output for predictions
        if 'mistral' in model.lower():
          mod_name = 'mistral'
        elif 'llama' in model.lower():
          mod_name = 'llama'
        else:
          mod_name = 'others'

        # Deal with inconsistencies in the output
        prediction_df['related_topics'] = prediction_df['related_topics'].apply(normalize_related_topics)

        try:
          prediction_df.to_parquet('prediction_df{}.parquet'.format(mod_name))
        except:
          prediction_df.to_excel('prediction_df{}.xlsx'.format(mod_name), index = False)

        # append to this final prediction df
        final_prediction_df = pd.concat((final_prediction_df, prediction_df), ignore_index = True)
        print('Prediction done for model {}: {}'.format(index+1, mod_name))

    return final_prediction_df

In [5]:
import random

def generate_random_list(max_num, list_size=3000, seed=1):
    """
    Generate a list of unique random integers from 0 to max_num - 1 (no duplicates).

    Parameters:
        max_num (int): The exclusive upper limit for random numbers.
        list_size (int): Number of random integers to generate.
        seed (int): Random seed for reproducibility.

    Returns:
        List[int]: A list of unique random integers.
    """
    if list_size > max_num:
        raise ValueError("list_size cannot be greater than max_num when sampling without replacement.")

    random.seed(seed)
    return random.sample(range(0, max_num), list_size)


In [ ]:
# Load in the sampled data
final_news_df_sampled = pd.read_parquet('final_cleaned_news_data_sampled.parquet', engine = 'pyarrow')

# Take only those with ner_index
# Get the clean_full_text columns as articles
articles = final_news_df_sampled['clean_full_text'].tolist()

final_prediction_df = classify_by_llm(articles)

final_prediction_df

Loading model: unsloth/mistral-7b-instruct-v0.3-bnb-4bit
==((====))==  Unsloth 2025.3.19: Fast Mistral patching. Transformers: 4.51.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
1 articles processed
51 articles processed
101 articles processed
151 articles processed
201 articles processed
251 articles processed
301 articles processed
351 articles processed
401 articles processed
451 articles processed
501 articles processed
551 articles processed
601 articles processed
651 articles processed
701 articles processed
751 articles processed
801 articles processed
851 articles processed
901 articles processed
951 articles processed
1001 articles processed
105

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.7k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

1 articles processed
51 articles processed
101 articles processed
151 articles processed
201 articles processed
251 articles processed
301 articles processed
351 articles processed
401 articles processed
451 articles processed
501 articles processed
551 articles processed
601 articles processed
651 articles processed
701 articles processed
751 articles processed
801 articles processed
851 articles processed
901 articles processed
951 articles processed
1001 articles processed
1051 articles processed
1101 articles processed
1151 articles processed
1201 articles processed
1251 articles processed
1301 articles processed
1351 articles processed
1401 articles processed
1451 articles processed
1501 articles processed
1551 articles processed
1601 articles processed
1651 articles processed
1701 articles processed
1751 articles processed
1801 articles processed
1851 articles processed
1901 articles processed
1951 articles processed
2001 articles processed
2051 articles processed
2101 articles p

In [ ]:
# Load in the sampled data
final_news_df_sampled = pd.read_parquet('final_cleaned_news_data_sampled.parquet', engine = 'pyarrow')

# Take only those with ner_index
# Get the clean_full_text columns as articles
articles = final_news_df_sampled['description'].tolist()

final_prediction_df = classify_by_llm(articles)

Loading model: unsloth/mistral-7b-instruct-v0.3-bnb-4bit
==((====))==  Unsloth 2025.3.19: Fast Mistral patching. Transformers: 4.51.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
1 articles processed
51 articles processed
101 articles processed
151 articles processed
201 articles processed
251 articles processed
301 articles processed
351 articles processed
401 articles processed
451 articles processed
501 articles processed
551 articles processed
601 articles processed
651 articles processed
701 articles processed
751 articles processed
801 articles processed
851 articles processed
901 articles processed
951 articles processed
1001 articles processed
105